## YT transcript to clickable links, using reportlab package

This can also be done using the fpdf lib, which I'm already using for html-->pdf

In [ ]:
from youtube_transcript_api import YouTubeTranscriptApi
from reportlab.lib.pagesizes import letter
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.colors import blue
from reportlab.lib.enums import TA_JUSTIFY
import re

def format_time(seconds):
    minutes, seconds = divmod(seconds, 60)
    hours, minutes = divmod(minutes, 60)
    return f"{int(hours):02d}:{int(minutes):02d}:{int(seconds):02d}"

def create_pdf(transcript, video_id):
    doc = SimpleDocTemplate("transcript.pdf", pagesize=letter)
    styles = getSampleStyleSheet()
    story = []

    link_style = ParagraphStyle(
        'LinkStyle',
        parent=styles['Normal'],
        textColor=blue,
        underline=True
    )

    text_style = ParagraphStyle(
        'TextStyle',
        parent=styles['Normal'],
        alignment=TA_JUSTIFY
    )

    for entry in transcript:
        timestamp = format_time(entry['start'])
        text = entry['text']
        
        time_link = f'<a href="https://www.youtube.com/watch?v={video_id}&t={int(entry["start"])}s" color="blue">{timestamp}</a>'
        
        paragraph = Paragraph(f"{time_link}: {text}", text_style)
        story.append(paragraph)
        story.append(Spacer(1, 6))

    doc.build(story)

def main():
    video_url = "https://www.youtube.com/watch?v=TaG2zCVotxE"
    video_id = re.search(r"v=([^&]+)", video_url).group(1)

    try:
        transcript = YouTubeTranscriptApi.get_transcript(video_id)
        create_pdf(transcript, video_id)
        print("Transcript has been downloaded and converted to PDF successfully.")
    except Exception as e:
        print(f"An error occurred: {str(e)}")

if __name__ == "__main__":
    main()